# 03 — Risk Scoring Model
> NovaCred Bank | Loan Default Risk Analysis

**Goal:** Train a logistic regression classifier to predict loan default probability, evaluate performance, assign risk tiers, and export scored data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report,
    ConfusionMatrixDisplay, precision_recall_curve, average_precision_score
)

df = pd.read_csv('../data/cleaned/loan_cleaned.csv')
print(f'Loaded {len(df):,} rows  |  Default rate: {df["default"].mean():.2%}')

## 1. Feature Selection & Train/Test Split

In [ ]:
FEATURES = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'loan_to_income', 'grade_num']

model_df = df[FEATURES + ['default']].dropna()
X = model_df[FEATURES]
y = model_df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Class balance (test): {y_test.value_counts(normalize=True).round(3).to_dict()}')

## 2. Scale Features & Train Model

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',   # handles class imbalance
    solver='lbfgs',
    random_state=42
)
model.fit(X_train_sc, y_train)

# Cross-validated AUC
cv_scores = cross_val_score(model, X_train_sc, y_train,
                             cv=StratifiedKFold(5), scoring='roc_auc')
print(f'CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 3. Model Evaluation

In [ ]:
y_pred_proba = model.predict_proba(X_test_sc)[:, 1]
y_pred       = model.predict(X_test_sc)

roc_auc = roc_auc_score(y_test, y_pred_proba)
avg_prec = average_precision_score(y_test, y_pred_proba)

print(f'Test ROC-AUC:          {roc_auc:.4f}')
print(f'Average Precision:     {avg_prec:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Fully Paid','Charged Off']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_pred_proba)
axes[1].plot(rec, prec, color='crimson', lw=2, label=f'Avg Prec = {avg_prec:.3f}')
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Fully Paid', 'Charged Off'],
    cmap='Blues', ax=axes[2]
)
axes[2].set_title('Confusion Matrix')

plt.suptitle('NovaCred Bank — Loan Default Risk Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/06_model_evaluation.png', dpi=150)
plt.show()

## 4. Feature Importance (Coefficients)

In [ ]:
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Feature Coefficients\n(positive = higher default risk)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('../reports/figures/07_feature_coefficients.png', dpi=150)
plt.show()
print(coef_df.to_string(index=False))

## 5. Assign Risk Scores & Tiers

In [ ]:
df_scored = df[FEATURES].dropna().copy()
X_all_sc  = scaler.transform(df_scored)

df_scored['risk_score'] = model.predict_proba(X_all_sc)[:, 1]
df_scored['risk_tier']  = pd.cut(
    df_scored['risk_score'],
    bins=[0, 0.30, 0.60, 1.0],
    labels=['Low', 'Medium', 'High']
)

tier_summary = df_scored['risk_tier'].value_counts().reset_index()
tier_summary.columns = ['Risk Tier', 'Count']
tier_summary['% of Portfolio'] = (tier_summary['Count'] / len(df_scored) * 100).round(1)
print(tier_summary.to_string(index=False))

df_scored.to_csv('../data/cleaned/loan_with_risk.csv', index=False)
print('\nScored dataset saved → data/cleaned/loan_with_risk.csv ✅')

## 6. Risk Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df_scored['risk_score'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(0.30, color='orange', linestyle='--', label='Low/Med boundary')
axes[0].axvline(0.60, color='red',    linestyle='--', label='Med/High boundary')
axes[0].set_title('Risk Score Distribution')
axes[0].set_xlabel('Risk Score')
axes[0].legend()

tier_counts = df_scored['risk_tier'].value_counts().reindex(['Low','Medium','High'])
axes[1].pie(tier_counts, labels=tier_counts.index,
            colors=['#2ecc71','#f39c12','#e74c3c'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Portfolio Risk Tier Breakdown')

plt.suptitle('NovaCred Bank — Risk Score Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/08_risk_score_distribution.png', dpi=150)
plt.show()